In [2]:
from pathlib import Path
import asyncio
import logging
import re
from urllib.parse import urlparse

import dask.bag as db
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster, WorkerPlugin, get_worker
from dask.distributed import as_completed
from coiled import Cluster as CoiledCluster

from obstore.store import S3Store
import rustac
import duckdb
import s3fs


logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [3]:
class RustacStorePlugin(WorkerPlugin):
    def __init__(self, read_config, write_config):
        self.store_config = read_config
        self.write_store_config = write_config
        self.store = None
        self.writer_store = None
    
    def setup(self, worker):
        """Initialize the store when a worker starts"""
        bucket,prefix,region = self.store_config
        self.store = S3Store(bucket=bucket, prefix=prefix, region=region, client_options={"timeout":"4m"}, skip_signature=True)
        bucket,prefix,region = self.write_store_config
        self.writer_store = S3Store(bucket=bucket, prefix=prefix, region=region, client_options={"timeout":"8m"})
        print("Initialized rustac store")
    
    def teardown(self, worker):
        """Cleanup when worker shuts down"""
        if self.store:
            print("Cleaned up rustac store")
    
    def get_store(self):
        if self.store is None:
            raise RuntimeError("Store not initialized")
        return self.store


def create_dask_cluster(environment="local",
                        source_store_opts=None,
                        output_store_opts=None,
                        n_workers=4,
                        threads_per_worker=1,
                        cloud_opts={}):

    
    if "client" in locals() and "cluster" in locals():
        return (client, cluster)
    else:
        if environment=="local":
            print("Creating new local Dask client")
            cluster = LocalCluster(n_workers=n_workers,
                                 threads_per_worker=threads_per_worker,
                                 silence_logs=logging.ERROR)
        else:
            print("Creating new Coiled Dask client")
            cluster = CoiledCluster(n_workers=n_workers,
                                    **cloud_opts)
        cluster.send_private_envs(
            {
                "AWS_SECRET_ACCESS_KEY": "***REMOVED***",
                "AWS_ACCESS_KEY_ID": "***REMOVED***"
            }
        )
        client = Client(cluster)
        obstore_plugin = RustacStorePlugin(source_store_opts, output_store_opts)
        client.register_plugin(obstore_plugin, name='rustac_store')
        return (client, cluster)

In [4]:
source_store = (
    "its-live-data",
    "test-space/cloud-experiments/catalog/landsat-consolidated-patched/",
    "us-west-2"
)

output_store = (
    "its-live-data",
    "test-space/cloud-experiments/catalog/geoparquet/landsatOLI/",
    "us-west-2"
)

cloud_opts = {
    "region": "us-west-2",
    # "scheduler_vm_types": [""]
    "worker_vm_types": ["c6a.2xlarge"],
    "spot_policy":"on-demand",
}

client, cluster = create_dask_cluster(
    environment="cloud",
    source_store_opts=source_store,
    output_store_opts=output_store,
    n_workers=4,
    threads_per_worker=64,
    cloud_opts=cloud_opts)
client

Creating new Coiled Dask client


Output()

INFO:coiled.package_sync:Resolving your local cryoforge-dev Python environment...
INFO:coiled.software_utils:No username or password found for https://conda.anaconda.org/conda-forge
INFO:coiled:Backing off _do_request(...) for 0.1s (<ClientResponse(https://cloud.coiled.io/api/v2/software-environment/approximate-packages) [504 Gateway Timeout]>
<CIMultiDictProxy('Content-Length': '941', 'Connection': 'keep-alive', 'Date': 'Wed, 11 Jun 2025 05:06:01 GMT', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 05ecb79dbd3bc8a5c99fa616e7de5b48.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'ORD58-P2', 'X-Amz-Cf-Id': 'bbO6Y8IiPZpeYtD6SRcUZfzTPxR3w7JDz_6DAQAq9vYn67_Y91H2cA==')>
)


╭──────────────────────────────── Package Info ────────────────────────────────╮
│                      ╷                                                       │
│   Package            │ Note                                                  │
│ ╶────────────────────┼─────────────────────────────────────────────────────╴ │
│   coiled_local_tests │ Source wheel built from                               │
│                      │ ~/nsidc/itslive/cryoforge/tests                       │
│   cryoforge          │ Wheel built from ~/nsidc/itslive/cryoforge            │
│                      ╵                                                       │
╰──────────────────────────────────────────────────────────────────────────────╯

INFO:coiled:Creating Cluster (name: nasa-openscapes-f43c9eef, https://cloud.coiled.io/clusters/931979 ). This usually takes 1-2 minutes...


Output()

<Client: 'tls://10.0.23.181:8786' processes=3 threads=24, memory=44.18 GiB>

In [5]:

async def async_process_batch(paths_batch, destination, semaphore_limit=10):
    """Process a batch of paths with async concurrency within the batch"""
    semaphore = asyncio.Semaphore(semaphore_limit)
    
    async def process_single_path(path):
        async with semaphore:
            try:
                # Get the plugin from the worker
                worker = get_worker()
                plugin = worker.plugins['rustac_store']
                store = plugin.store
                writer_store = plugin.writer_store
                
                value = await rustac.read(str(path), store=store)
                dest_path = path.replace(".ndjson", ".parquet")
                
                if value["type"] == "Feature":
                    await rustac.write(str(dest_path), [value], format="parquet", store=writer_store)
                else:
                    assert value["type"] == "FeatureCollection"
                    await rustac.write(str(dest_path), value, format="parquet", store=writer_store)
                
                return {'status': 'success', 'path': path}
                
            except Exception as e:
                logger.error(f"Failed to process {path}: {str(e)}")
                return {'status': 'failed', 'path': path, 'error': str(e)}
    
    # Process all paths in this batch concurrently
    tasks = [process_single_path(path) for path in paths_batch]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Handle any exceptions that were returned
    processed_results = []
    for result in results:
        if isinstance(result, Exception):
            processed_results.append({
                'status': 'failed', 
                'path': 'unknown', 
                'error': str(result)
            })
        else:
            processed_results.append(result)
    
    return processed_results

def run_async_batch(paths_batch, destination, semaphore_limit=10):
    """Wrapper to run async batch processing in a worker"""
    return asyncio.run(async_process_batch(paths_batch, destination, semaphore_limit))

def process_files_with_async_batches(client, paths, destination, batch_size=50, semaphore_limit=10, progress=True):
    """
    Process files using Dask workers, with async concurrency within each batch
    
    Architecture:
    - Dask distributes batches across workers (e.g., 4 workers get 4 different batches)
    - Each worker creates ONE event loop and processes its batch of paths concurrently
    - Within each batch, semaphore controls concurrent async operations
    
    Parameters:
    - batch_size: Number of paths per worker (e.g., 50 means each worker gets 50 paths)
    - semaphore_limit: Max concurrent async operations within each worker's batch
    """
    
    # Split paths into batches for distribution across workers
    batches = [paths[i:i + batch_size] for i in range(0, len(paths), batch_size)]
    
    print(f"Processing {len(paths)} paths in {len(batches)} batches")
    print(f"Each worker will process up to {batch_size} paths concurrently (limited by semaphore: {semaphore_limit})")
    
    # Submit each batch to a worker - each worker gets ONE batch and creates ONE event loop
    futures = [
        client.submit(run_async_batch, batch, destination, semaphore_limit)
        for batch in batches
    ]
    
    if progress:
        import tqdm
        all_results = []
        with tqdm.tqdm(total=len(paths), desc="Processing files") as pbar:
            for future in as_completed(futures):
                batch_results = future.result()
                all_results.extend(batch_results)
                pbar.update(len(batch_results))
        return all_results
    else:
        batch_results = client.gather(futures)
        # Flatten the results
        return [result for batch in batch_results for result in batch]

In [6]:
year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')


source_store = S3Store(
    bucket="its-live-data", prefix="test-space/cloud-experiments/catalog/landsat-consolidated-patched/", region="us-west-2", skip_signature=True
)

paths = []
sizes = []
for list_stream in source_store.list():
    for object_meta in list_stream:
        if year_file_re.match(object_meta["path"]):
            paths.append(object_meta["path"])
            sizes.append(object_meta["size"])
print(len(paths))

5512


In [7]:
paths[0]

'N20E080/1987.ndjson'

In [8]:
destination=""
results = process_files_with_async_batches(
    client, 
    paths, 
    destination, 
    batch_size=350,  # Paths per worker
    semaphore_limit=15,  # Concurrent async ops per worker
    progress=True
)

Processing 5512 paths in 16 batches
Each worker will process up to 350 paths concurrently (limited by semaphore: 15)


Processing files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5512/5512 [06:55<00:00, 13.27it/s]


In [ ]:
failed = [r["path"] for f in results]
len(failed)

In [11]:
for r in results:
    if r["status"] != "success":
        print(r)

In [13]:
results[0]

{'status': 'success', 'path': 'N70W060/1995.ndjson'}

In [ ]:
import duckdb


destination = Path("./data/geoparquet/new")

duckdb.sql(f"select count(*) from read_parquet('{destination}/**/*.parquet')")

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/new")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
import os.path
import humanize

destination = Path("./data/geoparquet/sentinel1")

count = 0
size = 0
for path in destination.glob("**/*.parquet"):
    count += 1
    size += os.path.getsize(path)

print(f"The {count} stac-geoparquet files are {humanize.naturalsize(size)}")
print(f"That's {100 * size / sum(sizes):.2f}% of the original size")

In [ ]:
# Debug what's happening with path parts
from pathlib import Path
from urllib.parse import urlparse

# Example path
path = "s3://its-live-data/test-space/stac-catalog/landsatOLI/v02/N30E100/2013.ndjson"

url_path = urlparse(path).path
print(f"url_path: {url_path}")

source_path = Path(url_path)
print(f"source_path.parts: {source_path.parts}")
print(f"Last 4 parts: {source_path.parts[-4:]}")

preserved_path = Path(*source_path.parts[-4:])
print(f"preserved_path: {preserved_path}")

# The fix - skip the root '/' by filtering it out
non_root_parts = [part for part in source_path.parts if part != '/']
print(f"non_root_parts: {non_root_parts}")
print(f"Last 4 non-root parts: {non_root_parts[-4:]}")

preserved_path_fixed = Path(*non_root_parts[-4:])
print(f"preserved_path_fixed: {preserved_path_fixed}")

In [ ]:


paths = [
    'landsatOLI/v02/N20E080/1987.ndjson',
    'landsatOLI/v02/N20E080/',
    'landsatOLI/v02/N20E080/README.txt',
    'landsatOLI/v02/S10W100/2003.ndjson'
]

filtered = [p for p in paths if year_file_re.match(p)]
print(filtered)